In [1]:
import os
import sys
import json
import random
import math
import copy
import time
import csv
import warnings

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import cv2

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    ConcatDataset,
    WeightedRandomSampler,
)

from datasets import load_dataset

from transformers import (
    Sam2Model,
    Sam2Processor,
)

warnings.filterwarnings("ignore")

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

Python: 3.10.12 (main, Jun 22 2026, 18:55:27) [GCC 11.4.0]
PyTorch: 2.12.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 4090
CUDA: 13.0


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MODEL_NAME = "facebook/sam2.1-hiera-tiny"

# Same training setup as your original SAM2 experiment
BATCH_SIZE = 1
NUM_EPOCHS = 10
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

NUM_WORKERS = 0

# Early stopping is useful especially for EWI1S
EARLY_STOPPING_PATIENCE = 3

# Prompt perturbation used only during training
BOX_JITTER_FRACTION = 0.05

# Evaluation threshold
MASK_THRESHOLD = 0.5

# Joint-training balancing
# Each dataset contributes approximately this many samples per epoch.
JOINT_SAMPLES_PER_DATASET = 500

PROJECT_ROOT = Path("/home/zahranadeem/thesis")

DOMAIN_FT_ROOT = (
    PROJECT_ROOT / "sam2_domain_finetuning"
)

DOMAIN_FT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("Device:", device)
print("Model:", MODEL_NAME)
print("Output directory:", DOMAIN_FT_ROOT)

Device: cuda
Model: facebook/sam2.1-hiera-tiny
Output directory: /home/zahranadeem/thesis/sam2_domain_finetuning


In [3]:
# ============================================================
# Existing VegAnn SAM2 checkpoint
# ============================================================

VEGANN_SAM2_CHECKPOINT = (
    PROJECT_ROOT
    / "sam2_vegann_checkpoints"
    / "best_sam2_vegann_mask_decoder.pt"
)


# ============================================================
# ReLeaf
# ============================================================

RELEAF_ROOT = (
    PROJECT_ROOT / "CropAndWeedAndLeaf"
)

RELEAF_REPO = (
    PROJECT_ROOT / "releaf"
)

RELEAF_IMAGES_DIR = (
    RELEAF_ROOT / "images"
)

RELEAF_LABELS_DIR = (
    RELEAF_ROOT / "labels"
)

RELEAF_TEST_DIR = (
    RELEAF_REPO
    / "data"
    / "CWL"
    / "test"
)


# ============================================================
# WeedsGalore
# ============================================================

WEEDSGALORE_ROOT = (
    PROJECT_ROOT / "WeedsGalore_dataset"
)

WEEDS_IMAGE_DIR = (
    WEEDSGALORE_ROOT / "data"
)

WEEDS_MASK_DIR = (
    WEEDSGALORE_ROOT
    / "fields"
    / "ground_truth_semantic"
)


# ============================================================
# EWI1S
# ============================================================

EWIS1_ROOT = (
    PROJECT_ROOT / "EWIS1"
)


# Search recursively because your extracted archive has
# nested "images/images" and "masks/masks" directories.

def find_nested_dataset_dir(
    root,
    dirname,
    extension=".png"
):
    candidates = []

    for p in root.rglob(dirname):
        if not p.is_dir():
            continue

        files = [
            x for x in p.iterdir()
            if x.is_file()
            and x.suffix.lower() == extension.lower()
        ]

        if len(files) > 0:
            candidates.append(
                (len(files), p)
            )

    if not candidates:
        return None

    candidates.sort(
        reverse=True,
        key=lambda x: x[0]
    )

    return candidates[0][1]


EWIS1_IMAGE_DIR = find_nested_dataset_dir(
    EWIS1_ROOT,
    "images",
    ".png"
)

EWIS1_MASK_DIR = find_nested_dataset_dir(
    EWIS1_ROOT,
    "masks",
    ".png"
)


# ============================================================
# Print paths
# ============================================================

print("VegAnn SAM2 checkpoint:")
print(VEGANN_SAM2_CHECKPOINT)

print("\nReLeaf images:")
print(RELEAF_IMAGES_DIR)

print("ReLeaf labels:")
print(RELEAF_LABELS_DIR)

print("ReLeaf test:")
print(RELEAF_TEST_DIR)

print("\nWeedsGalore images:")
print(WEEDS_IMAGE_DIR)

print("WeedsGalore masks:")
print(WEEDS_MASK_DIR)

print("\nEWI1S images:")
print(EWIS1_IMAGE_DIR)

print("EWI1S masks:")
print(EWIS1_MASK_DIR)

VegAnn SAM2 checkpoint:
/home/zahranadeem/thesis/sam2_vegann_checkpoints/best_sam2_vegann_mask_decoder.pt

ReLeaf images:
/home/zahranadeem/thesis/CropAndWeedAndLeaf/images
ReLeaf labels:
/home/zahranadeem/thesis/CropAndWeedAndLeaf/labels
ReLeaf test:
/home/zahranadeem/thesis/releaf/data/CWL/test

WeedsGalore images:
/home/zahranadeem/thesis/WeedsGalore_dataset/data
WeedsGalore masks:
/home/zahranadeem/thesis/WeedsGalore_dataset/fields/ground_truth_semantic

EWI1S images:
/home/zahranadeem/thesis/EWIS1/EWIS1 Maize, Sorghum & Weed Dataset/images/images
EWI1S masks:
/home/zahranadeem/thesis/EWIS1/EWIS1 Maize, Sorghum & Weed Dataset/masks/masks


In [4]:
paths_to_check = {
    "VegAnn SAM2 checkpoint": VEGANN_SAM2_CHECKPOINT,
    "ReLeaf images": RELEAF_IMAGES_DIR,
    "ReLeaf labels": RELEAF_LABELS_DIR,
    "WeedsGalore images": WEEDS_IMAGE_DIR,
    "WeedsGalore masks": WEEDS_MASK_DIR,
    "EWI1S images": EWIS1_IMAGE_DIR,
    "EWI1S masks": EWIS1_MASK_DIR,
}

for name, path in paths_to_check.items():
    print(
        f"{name:30s}: "
        f"{'OK' if path is not None and path.exists() else 'MISSING'}"
    )

if not VEGANN_SAM2_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Existing VegAnn SAM2 checkpoint not found:\n"
        f"{VEGANN_SAM2_CHECKPOINT}"
    )

VegAnn SAM2 checkpoint        : OK
ReLeaf images                 : OK
ReLeaf labels                 : OK
WeedsGalore images            : OK
WeedsGalore masks             : OK
EWI1S images                  : OK
EWI1S masks                   : OK


In [5]:
#inspecting external datasets 
def count_files(directory, extensions):
    if directory is None or not directory.exists():
        return 0

    return sum(
        1
        for p in directory.rglob("*")
        if p.is_file()
        and p.suffix.lower() in extensions
    )


IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".tif",
    ".tiff",
    ".bmp"
}

MASK_EXTENSIONS = IMAGE_EXTENSIONS


print(
    "ReLeaf images:",
    count_files(RELEAF_IMAGES_DIR, IMAGE_EXTENSIONS)
)

print(
    "ReLeaf labels:",
    count_files(RELEAF_LABELS_DIR, {".txt"})
)

print(
    "WeedsGalore images:",
    count_files(WEEDS_IMAGE_DIR, IMAGE_EXTENSIONS)
)

print(
    "WeedsGalore masks:",
    count_files(WEEDS_MASK_DIR, MASK_EXTENSIONS)
)

print(
    "EWI1S images:",
    count_files(EWIS1_IMAGE_DIR, IMAGE_EXTENSIONS)
)

print(
    "EWI1S masks:",
    count_files(EWIS1_MASK_DIR, MASK_EXTENSIONS)
)

ReLeaf images: 345
ReLeaf labels: 345
WeedsGalore images: 156
WeedsGalore masks: 156
EWI1S images: 88
EWI1S masks: 88


In [6]:
#load vegann and split 
vegann = load_dataset(
    "simonMadec/VegAnn"
)

print(vegann)
print("Columns:", vegann["train"].column_names)
print("Total images:", len(vegann["train"]))

vegann_split = vegann["train"].train_test_split(
    test_size=0.20,
    seed=SEED
)

vegann_train_hf = vegann_split["train"]
vegann_val_hf = vegann_split["test"]

print(
    "VegAnn training:",
    len(vegann_train_hf)
)

print(
    "VegAnn validation:",
    len(vegann_val_hf)
)

DatasetDict({
    train: Dataset({
        features: ['image', 'mask', 'System', 'Orientation', 'latitude', 'longitude', 'date', 'LocAcc', 'Species', 'Owner', 'Dataset-Name', 'TVT-split1', 'TVT-split2', 'TVT-split3', 'TVT-split4', 'TVT-split5'],
        num_rows: 3775
    })
})
Columns: ['image', 'mask', 'System', 'Orientation', 'latitude', 'longitude', 'date', 'LocAcc', 'Species', 'Owner', 'Dataset-Name', 'TVT-split1', 'TVT-split2', 'TVT-split3', 'TVT-split4', 'TVT-split5']
Total images: 3775
VegAnn training: 3020
VegAnn validation: 755


In [7]:
def binary_mask_from_array(mask):
    """
    Convert any mask-like object into uint8 binary mask.
    Any non-zero pixel is foreground.
    """
    
    if isinstance(mask, Image.Image):
        mask = np.array(mask)

    elif torch.is_tensor(mask):
        mask = mask.detach().cpu().numpy()

    mask = np.asarray(mask)

    # Remove unnecessary channel dimensions
    while mask.ndim > 2:
        mask = mask[..., 0]

    binary = (
        mask > 0
    ).astype(np.uint8)

    return binary


def mask_to_box(mask):
    """
    Compute [x1, y1, x2, y2] from a binary mask.
    Returns None for empty masks.
    """

    mask = binary_mask_from_array(mask)

    ys, xs = np.where(mask > 0)

    if len(xs) == 0:
        return None

    x1 = int(xs.min())
    y1 = int(ys.min())

    x2 = int(xs.max())
    y2 = int(ys.max())

    return np.array(
        [x1, y1, x2, y2],
        dtype=np.int32
    )

In [8]:
#BOX PERTURBATION 
def jitter_box(
    box,
    image_width,
    image_height,
    fraction=BOX_JITTER_FRACTION
):
    """
    Randomly shift/expand/shrink a GT box slightly.
    Used only during training.
    """

    x1, y1, x2, y2 = [
        int(v) for v in box
    ]

    box_width = max(
        1,
        x2 - x1
    )

    box_height = max(
        1,
        y2 - y1
    )

    max_dx = int(
        box_width * fraction
    )

    max_dy = int(
        box_height * fraction
    )

    dx1 = random.randint(
        -max_dx,
        max_dx
    )

    dy1 = random.randint(
        -max_dy,
        max_dy
    )

    dx2 = random.randint(
        -max_dx,
        max_dx
    )

    dy2 = random.randint(
        -max_dy,
        max_dy
    )

    x1 = x1 + dx1
    y1 = y1 + dy1
    x2 = x2 + dx2
    y2 = y2 + dy2

    x1 = max(
        0,
        min(x1, image_width - 1)
    )

    y1 = max(
        0,
        min(y1, image_height - 1)
    )

    x2 = max(
        x1 + 1,
        min(x2, image_width - 1)
    )

    y2 = max(
        y1 + 1,
        min(y2, image_height - 1)
    )

    return np.array(
        [x1, y1, x2, y2],
        dtype=np.int32
    )

In [9]:
#ReLeaf Laoder 
def find_matching_mask_or_label(
    directory,
    image_path
):
    """
    Find a file with the same stem.
    """

    candidates = []

    for suffix in [
        ".txt",
        ".png",
        ".jpg",
        ".jpeg",
        ".tif",
        ".tiff"
    ]:
        candidate = (
            directory
            / f"{image_path.stem}{suffix}"
        )

        if candidate.exists():
            candidates.append(candidate)

    return candidates[0] if candidates else None


def load_releaf_image_mask(
    image_path
):
    """
    ReLeaf:
    YOLO segmentation polygons -> binary vegetation mask.
    """

    image = Image.open(
        image_path
    ).convert("RGB")

    width, height = image.size

    mask = np.zeros(
        (height, width),
        dtype=np.uint8
    )

    label_path = (
        RELEAF_LABELS_DIR
        / f"{image_path.stem}.txt"
    )

    if not label_path.exists():
        return image, mask

    with open(
        label_path,
        "r"
    ) as f:
        lines = f.readlines()

    for line in lines:

        values = line.strip().split()

        if len(values) < 7:
            continue

        # Ignore first value = class id
        coords = [
            float(v)
            for v in values[1:]
        ]

        if len(coords) < 6:
            continue

        points = []

        for i in range(
            0,
            len(coords),
            2
        ):
            x = int(
                coords[i] * width
            )

            y = int(
                coords[i + 1] * height
            )

            points.append(
                [x, y]
            )

        points = np.asarray(
            points,
            dtype=np.int32
        )

        cv2.fillPoly(
            mask,
            [points],
            1
        )

    return image, mask

In [10]:
releaf_pairs = []

releaf_image_files = sorted(
    [
        p
        for p in RELEAF_IMAGES_DIR.iterdir()
        if p.is_file()
        and p.suffix.lower()
        in IMAGE_EXTENSIONS
    ]
)

for image_path in releaf_image_files:

    label_path = (
        RELEAF_LABELS_DIR
        / f"{image_path.stem}.txt"
    )

    if label_path.exists():

        releaf_pairs.append(
            {
                "image_path": image_path,
                "label_path": label_path,
                "dataset": "ReLeaf"
            }
        )

print(
    "ReLeaf image-label pairs:",
    len(releaf_pairs)
)

print(
    releaf_pairs[:3]
)

ReLeaf image-label pairs: 345
[{'image_path': PosixPath('/home/zahranadeem/thesis/CropAndWeedAndLeaf/images/ave-0041-0002-002.jpg'), 'label_path': PosixPath('/home/zahranadeem/thesis/CropAndWeedAndLeaf/labels/ave-0041-0002-002.txt'), 'dataset': 'ReLeaf'}, {'image_path': PosixPath('/home/zahranadeem/thesis/CropAndWeedAndLeaf/images/ave-0045-0011-012.jpg'), 'label_path': PosixPath('/home/zahranadeem/thesis/CropAndWeedAndLeaf/labels/ave-0045-0011-012.txt'), 'dataset': 'ReLeaf'}, {'image_path': PosixPath('/home/zahranadeem/thesis/CropAndWeedAndLeaf/images/ave-0048-0002-026.jpg'), 'label_path': PosixPath('/home/zahranadeem/thesis/CropAndWeedAndLeaf/labels/ave-0048-0002-026.txt'), 'dataset': 'ReLeaf'}]


In [11]:
releaf_official_test_names = set()

if RELEAF_TEST_DIR.exists():

    for p in RELEAF_TEST_DIR.iterdir():

        if (
            p.is_file()
            and p.suffix.lower()
            in IMAGE_EXTENSIONS
        ):
            releaf_official_test_names.add(
                p.name
            )

print(
    "Existing ReLeaf test images:",
    len(releaf_official_test_names)
)

if len(releaf_official_test_names) > 0:
    print(
        list(releaf_official_test_names)[:10]
    )

Existing ReLeaf test images: 345
['ave-0230-0006-003.jpg', 'ave-0291-0015-000.jpg', 'vwg-0091-0008-000.jpg', 'ave-0291-0011-000.jpg', 'vwg-0005-0005-000.jpg', 'ave-0048-0002-026.jpg', 'vwg-0520-0002-093.jpg', 'ave-0079-0002-002.jpg', 'vwg-0113-0012-004.jpg', 'ave-0291-0020-000.jpg']


In [12]:

releaf_official_test_names = {
    "image1.jpg", 
    "image2.jpg"
} 


In [13]:
releaf_official_test_names = set()


In [16]:
# ============================================================
# 1. Define the Deterministic Splitting Function
# ============================================================
def split_pairs(
    pairs,
    seed=42,
    train_fraction=0.80,
    val_fraction=0.10
):
    """
    Deterministic 80/10/10 split.
    """
    pairs = list(pairs)
    rng = random.Random(seed)
    rng.shuffle(pairs)

    n = len(pairs)
    n_train = int(train_fraction * n)
    n_val = int(val_fraction * n)

    train = pairs[:n_train]
    val = pairs[n_train:n_train + n_val]
    test = pairs[n_train + n_val:]

    return train, val, test


# ============================================================
# 2. Split ReLeaf dataset into Train, Val, and Test
# ============================================================

# Filter out the official test names from the full pool first
releaf_pool_for_splitting = [
    p for p in releaf_pairs 
    if p["image_path"].name not in releaf_official_test_names
]

# Split the remaining pool using the function defined above
releaf_train_pairs, releaf_val_pairs, releaf_test_pairs = split_pairs(
    releaf_pool_for_splitting,
    seed=SEED
)

print("ReLeaf Split Complete:")
print("Train:", len(releaf_train_pairs))
print("Val  :", len(releaf_val_pairs))
print("Test :", len(releaf_test_pairs))


ReLeaf Split Complete:
Train: 276
Val  : 34
Test : 35


In [17]:
# ============================================================
# PHASE 0 — ReLeaf 10% LABELLED TRAINING SUBSET
# ============================================================

PHASE0_RELEAF_IMAGES = 34

rng = random.Random(SEED)

# Keep the original full training pool
releaf_train_pairs_full = list(
    releaf_train_pairs
)

# Select exactly 34 labelled training images
releaf_train_pairs_phase0 = rng.sample(
    releaf_train_pairs_full,
    PHASE0_RELEAF_IMAGES
)

print("=" * 60)
print("PHASE 0 — ReLeaf")
print("=" * 60)

print(
    "Full training pool:",
    len(releaf_train_pairs_full)
)

print(
    "Selected labelled images:",
    len(releaf_train_pairs_phase0)
)

print(
    "Validation images:",
    len(releaf_val_pairs)
)

print(
    "Test images:",
    len(releaf_test_pairs)
)

print("=" * 60)

PHASE 0 — ReLeaf
Full training pool: 276
Selected labelled images: 34
Validation images: 34
Test images: 35


In [18]:
#WeedsGalore pair builder 
def find_mask_for_stem(
    mask_dir,
    stem
):
    """
    Search for a matching mask regardless of extension.
    """

    matches = list(
        mask_dir.glob(
            f"{stem}.*"
        )
    )

    matches = [
        p for p in matches
        if p.suffix.lower()
        in MASK_EXTENSIONS
    ]

    if matches:
        return matches[0]

    return None


weedsgalore_pairs = []

weed_image_files = sorted(
    [
        p
        for p in WEEDS_IMAGE_DIR.rglob("*")
        if p.is_file()
        and p.suffix.lower()
        in IMAGE_EXTENSIONS
    ]
)

for image_path in weed_image_files:

    mask_path = find_mask_for_stem(
        WEEDS_MASK_DIR,
        image_path.stem
    )

    if mask_path is not None:

        weedsgalore_pairs.append(
            {
                "image_path": image_path,
                "mask_path": mask_path,
                "dataset": "WeedsGalore"
            }
        )

print(
    "WeedsGalore matched pairs:",
    len(weedsgalore_pairs)
)

print(
    weedsgalore_pairs[:3]
)

WeedsGalore matched pairs: 156
[{'image_path': PosixPath('/home/zahranadeem/thesis/WeedsGalore_dataset/data/2023-05-25_0109.png'), 'mask_path': PosixPath('/home/zahranadeem/thesis/WeedsGalore_dataset/fields/ground_truth_semantic/2023-05-25_0109.png'), 'dataset': 'WeedsGalore'}, {'image_path': PosixPath('/home/zahranadeem/thesis/WeedsGalore_dataset/data/2023-05-25_0114.png'), 'mask_path': PosixPath('/home/zahranadeem/thesis/WeedsGalore_dataset/fields/ground_truth_semantic/2023-05-25_0114.png'), 'dataset': 'WeedsGalore'}, {'image_path': PosixPath('/home/zahranadeem/thesis/WeedsGalore_dataset/data/2023-05-25_0119.png'), 'mask_path': PosixPath('/home/zahranadeem/thesis/WeedsGalore_dataset/fields/ground_truth_semantic/2023-05-25_0119.png'), 'dataset': 'WeedsGalore'}]


In [19]:
#WeedsGalore loader + check mask values 
def load_weedsgalore_image_mask(
    item
):
    image = Image.open(
        item["image_path"]
    ).convert("RGB")

    mask = Image.open(
        item["mask_path"]
    )

    mask = binary_mask_from_array(
        mask
    )

    return image, mask


# Inspect a sample
image, mask = load_weedsgalore_image_mask(
    weedsgalore_pairs[0]
)

print("Image size:", image.size)
print("Mask shape:", mask.shape)
print("Mask values:", np.unique(mask))
print("Foreground pixels:", int(mask.sum()))

Image size: (600, 600)
Mask shape: (600, 600)
Mask values: [0 1]
Foreground pixels: 975


In [20]:
#EWIS1 pairs 
ewis1_pairs = []

ewis1_image_files = sorted(
    [
        p
        for p in EWIS1_IMAGE_DIR.iterdir()
        if p.is_file()
        and p.suffix.lower() == ".png"
    ]
)

for image_path in ewis1_image_files:

    mask_path = (
        EWIS1_MASK_DIR
        / image_path.name
    )

    if mask_path.exists():

        ewis1_pairs.append(
            {
                "image_path": image_path,
                "mask_path": mask_path,
                "dataset": "EWI1S"
            }
        )

print(
    "EWI1S matched pairs:",
    len(ewis1_pairs)
)

EWI1S matched pairs: 88


In [21]:
#EWIS1 loader 

def load_ewis1_image_mask(
    item
):
    image = Image.open(
        item["image_path"]
    ).convert("RGB")

    mask_rgba = np.array(
        Image.open(
            item["mask_path"]
        ).convert("RGBA")
    )

    rgb = mask_rgba[:, :, :3]
    alpha = mask_rgba[:, :, 3]

    # Official EWI1S weed color
    weed_pixels = (
        (rgb[:, :, 0] == 255)
        & (rgb[:, :, 1] == 127)
        & (rgb[:, :, 2] == 14)
        & (alpha > 0)
    )

    weed_mask = (
        weed_pixels
        .astype(np.uint8)
    )

    return image, weed_mask


# Verify sample
image, mask = load_ewis1_image_mask(
    ewis1_pairs[0]
)

print("Image size:", image.size)
print("Mask shape:", mask.shape)
print("Mask values:", np.unique(mask))
print("Weed pixels:", int(mask.sum()))

Image size: (5464, 3640)
Mask shape: (3640, 5464)
Mask values: [0 1]
Weed pixels: 48261


In [22]:
#Split weedsgalore and EWIS1 
def split_pairs(
    pairs,
    seed=42,
    train_fraction=0.80,
    val_fraction=0.10
):
    """
    Deterministic 80/10/10 split.
    """

    pairs = list(pairs)

    rng = random.Random(seed)

    rng.shuffle(pairs)

    n = len(pairs)

    n_train = int(
        train_fraction * n
    )

    n_val = int(
        val_fraction * n
    )

    train = pairs[
        :n_train
    ]

    val = pairs[
        n_train:n_train + n_val
    ]

    test = pairs[
        n_train + n_val:
    ]

    return train, val, test


weedsgalore_train_pairs, weedsgalore_val_pairs, weedsgalore_test_pairs = (
    split_pairs(
        weedsgalore_pairs,
        seed=SEED
    )
)

ewis1_train_pairs, ewis1_val_pairs, ewis1_test_pairs = (
    split_pairs(
        ewis1_pairs,
        seed=SEED
    )
)


print("WeedsGalore")
print("Train:", len(weedsgalore_train_pairs))
print("Val  :", len(weedsgalore_val_pairs))
print("Test :", len(weedsgalore_test_pairs))

print("\nEWI1S")
print("Train:", len(ewis1_train_pairs))
print("Val  :", len(ewis1_val_pairs))
print("Test :", len(ewis1_test_pairs))

WeedsGalore
Train: 124
Val  : 15
Test : 17

EWI1S
Train: 70
Val  : 8
Test : 10


In [23]:
# Remove empty masks from training/validation 
def filter_nonempty_pairs(
    pairs,
    loader_function
):
    valid = []
    empty = []

    for item in tqdm(
        pairs,
        desc=f"Checking {pairs[0]['dataset'] if pairs else 'dataset'}"
    ):
        _, mask = loader_function(
            item
        )

        if mask.sum() > 0:
            valid.append(item)
        else:
            empty.append(item)

    return valid, empty

In [24]:
#Filter external datasets 
releaf_train_pairs, releaf_train_empty = (
    filter_nonempty_pairs(
        releaf_train_pairs,
        lambda item: load_releaf_image_mask(
            item["image_path"]
        )
    )
)

releaf_val_pairs, releaf_val_empty = (
    filter_nonempty_pairs(
        releaf_val_pairs,
        lambda item: load_releaf_image_mask(
            item["image_path"]
        )
    )
)


weedsgalore_train_pairs, weedsgalore_train_empty = (
    filter_nonempty_pairs(
        weedsgalore_train_pairs,
        load_weedsgalore_image_mask
    )
)

weedsgalore_val_pairs, weedsgalore_val_empty = (
    filter_nonempty_pairs(
        weedsgalore_val_pairs,
        load_weedsgalore_image_mask
    )
)


ewis1_train_pairs, ewis1_train_empty = (
    filter_nonempty_pairs(
        ewis1_train_pairs,
        load_ewis1_image_mask
    )
)

ewis1_val_pairs, ewis1_val_empty = (
    filter_nonempty_pairs(
        ewis1_val_pairs,
        load_ewis1_image_mask
    )
)


print("\nEmpty training/validation masks")

print(
    "ReLeaf:",
    len(releaf_train_empty),
    "train,",
    len(releaf_val_empty),
    "val"
)

print(
    "WeedsGalore:",
    len(weedsgalore_train_empty),
    "train,",
    len(weedsgalore_val_empty),
    "val"
)

print(
    "EWI1S:",
    len(ewis1_train_empty),
    "train,",
    len(ewis1_val_empty),
    "val"
)

Checking ReLeaf:   0%|          | 0/276 [00:00<?, ?it/s]

Checking ReLeaf:   0%|          | 0/34 [00:00<?, ?it/s]

Checking WeedsGalore:   0%|          | 0/124 [00:00<?, ?it/s]

Checking WeedsGalore:   0%|          | 0/15 [00:00<?, ?it/s]

Checking EWI1S:   0%|          | 0/70 [00:00<?, ?it/s]

Checking EWI1S:   0%|          | 0/8 [00:00<?, ?it/s]


Empty training/validation masks
ReLeaf: 0 train, 0 val
WeedsGalore: 0 train, 0 val
EWI1S: 0 train, 0 val


In [25]:
#UNIFIED DATASET CLASS 

class UnifiedSegmentationDataset(Dataset):

    def __init__(
        self,
        records,
        dataset_name,
        jitter=False
    ):
        self.records = records
        self.dataset_name = dataset_name
        self.jitter = jitter

    def __len__(self):
        return len(self.records)

    def _load_item(
        self,
        record
    ):

        # ----------------------------
        # VegAnn
        # ----------------------------
        if self.dataset_name == "VegAnn":

            hf_dataset = record["hf_dataset"]
            index = record["index"]

            sample = hf_dataset[index]

            image = sample["image"]
            mask = sample["mask"]

            if not isinstance(
                image,
                Image.Image
            ):
                image = Image.fromarray(
                    np.array(image)
                )

            mask = binary_mask_from_array(
                mask
            )

            return (
                image.convert("RGB"),
                mask
            )

        # ----------------------------
        # ReLeaf
        # ----------------------------
        elif self.dataset_name == "ReLeaf":

            return load_releaf_image_mask(
                record["image_path"]
            )

        # ----------------------------
        # WeedsGalore
        # ----------------------------
        elif self.dataset_name == "WeedsGalore":

            return load_weedsgalore_image_mask(
                record
            )

        # ----------------------------
        # EWI1S
        # ----------------------------
        elif self.dataset_name == "EWI1S":

            return load_ewis1_image_mask(
                record
            )

        else:
            raise ValueError(
                f"Unknown dataset: {self.dataset_name}"
            )

    def __getitem__(
        self,
        index
    ):

        record = self.records[index]

        image, mask = self._load_item(
            record
        )

        box = mask_to_box(
            mask
        )

        if box is None:
            raise RuntimeError(
                f"Empty mask encountered in "
                f"{self.dataset_name}"
            )

        width, height = image.size

        # Training:
        # slightly perturb GT box.
        # Validation:
        # exact GT box.
        if self.jitter:

            box_for_model = jitter_box(
                box,
                width,
                height
            )

        else:
            box_for_model = box.copy()

        return {
            "image": image,
            "mask": mask,
            "box": box_for_model,
            "gt_box": box,
            "index": index,
            "dataset": self.dataset_name
        }

In [26]:
# Cell 21 — Build VegAnn records, excluding empty masks

vegann_train_records = []

for i in range(len(vegann_train_hf)):
    sample = vegann_train_hf[i]
    mask = binary_mask_from_array(sample["mask"])

    if mask.sum() > 0:
        vegann_train_records.append({
            "hf_dataset": vegann_train_hf,
            "index": i
        })


vegann_val_records = []

for i in range(len(vegann_val_hf)):
    sample = vegann_val_hf[i]
    mask = binary_mask_from_array(sample["mask"])

    if mask.sum() > 0:
        vegann_val_records.append({
            "hf_dataset": vegann_val_hf,
            "index": i
        })


print("VegAnn usable train records:", len(vegann_train_records))
print("VegAnn usable val records:", len(vegann_val_records))

VegAnn usable train records: 2966
VegAnn usable val records: 746


In [27]:
#Create all individual datasets 
datasets_train = {}

datasets_val = {}

# VegAnn
datasets_train["VegAnn"] = UnifiedSegmentationDataset(
    vegann_train_records,
    "VegAnn",
    jitter=True
)

datasets_val["VegAnn"] = UnifiedSegmentationDataset(
    vegann_val_records,
    "VegAnn",
    jitter=False
)


# ReLeaf
datasets_train["ReLeaf"] = UnifiedSegmentationDataset(
    releaf_train_pairs,
    "ReLeaf",
    jitter=True
)

datasets_val["ReLeaf"] = UnifiedSegmentationDataset(
    releaf_val_pairs,
    "ReLeaf",
    jitter=False
)


# WeedsGalore
datasets_train["WeedsGalore"] = UnifiedSegmentationDataset(
    weedsgalore_train_pairs,
    "WeedsGalore",
    jitter=True
)

datasets_val["WeedsGalore"] = UnifiedSegmentationDataset(
    weedsgalore_val_pairs,
    "WeedsGalore",
    jitter=False
)


# EWI1S
datasets_train["EWI1S"] = UnifiedSegmentationDataset(
    ewis1_train_pairs,
    "EWI1S",
    jitter=True
)

datasets_val["EWI1S"] = UnifiedSegmentationDataset(
    ewis1_val_pairs,
    "EWI1S",
    jitter=False
)


for name in datasets_train:

    print(
        f"{name:15s} | "
        f"train={len(datasets_train[name]):5d} | "
        f"val={len(datasets_val[name]):5d}"
    )

VegAnn          | train= 2966 | val=  746
ReLeaf          | train=  276 | val=   34
WeedsGalore     | train=  124 | val=   15
EWI1S           | train=   70 | val=    8


In [28]:
#Test every dataset item before loading SAM2 
for name, dataset in datasets_train.items():

    item = dataset[0]

    print("\n", "=" * 60)
    print(name)

    print(
        "Image:",
        item["image"].size
    )

    print(
        "Mask:",
        item["mask"].shape,
        np.unique(item["mask"])
    )

    print(
        "GT box:",
        item["gt_box"]
    )

    print(
        "Training box:",
        item["box"]
    )


VegAnn
Image: (512, 512)
Mask: (512, 512) [0 1]
GT box: [ 45   0 454 386]
Training box: [ 65   0 435 384]

ReLeaf
Image: (272, 199)
Mask: (199, 272) [0 1]
GT box: [  0   0 271 198]
Training box: [  0   0 262 192]

WeedsGalore
Image: (600, 600)
Mask: (600, 600) [0 1]
GT box: [  0   0 599 599]
Training box: [ 14  18 599 599]

EWI1S
Image: (5464, 3640)
Mask: (3640, 5464) [0 1]
GT box: [   0    0 5463 3639]
Training box: [   0  121 5463 3474]


In [29]:
#SAM2 collate and PIL cant be stacked by default PyTorch collate 
def sam2_collate_fn(batch):

    return {
        "images": [
            item["image"]
            for item in batch
        ],

        "masks": [
            item["mask"]
            for item in batch
        ],

        "boxes": [
            item["box"]
            for item in batch
        ],

        "gt_boxes": [
            item["gt_box"]
            for item in batch
        ],

        "indices": [
            item["index"]
            for item in batch
        ],

        "datasets": [
            item["dataset"]
            for item in batch
        ]
    }


In [30]:
processor = Sam2Processor.from_pretrained(
    MODEL_NAME
)

print(
    "SAM2 processor loaded."
)

SAM2 processor loaded.


In [31]:
def prepare_sam2_batch(
    batch
):

    images = batch["images"]
    masks = batch["masks"]
    boxes = batch["boxes"]

    input_boxes = [
        [box.tolist()]
        for box in boxes
    ]

    inputs = processor(
        images=images,
        segmentation_maps=masks,
        input_boxes=input_boxes,
        return_tensors="pt"
    )

    inputs = {
        key: (
            value.to(device)
            if torch.is_tensor(value)
            else value
        )
        for key, value in inputs.items()
    }

    return inputs

In [32]:
#Loss functions 
bce_loss_fn = nn.BCEWithLogitsLoss()


def dice_loss(
    logits,
    targets,
    smooth=1.0
):

    probabilities = torch.sigmoid(
        logits
    )

    probabilities = probabilities.flatten(1)

    targets = targets.flatten(1)

    intersection = (
        probabilities * targets
    ).sum(dim=1)

    dice = (
        2.0 * intersection + smooth
    ) / (
        probabilities.sum(dim=1)
        + targets.sum(dim=1)
        + smooth
    )

    return 1.0 - dice.mean()


def combined_segmentation_loss(
    logits,
    targets
):

    bce = bce_loss_fn(
        logits,
        targets
    )

    dice = dice_loss(
        logits,
        targets
    )

    total = (
        0.5 * bce
        + 0.5 * dice
    )

    return total, bce, dice

In [33]:
#extract SAM2 mask logits 
def extract_single_mask_logits(
    pred_masks
):
    """
    Convert SAM2 mask output to [B, H, W].
    """

    logits = pred_masks

    while logits.ndim > 3:
        logits = logits[:, 0]

    return logits

In [34]:
#Mtrics 
def compute_batch_metrics(
    logits,
    targets,
    threshold=0.5,
    epsilon=1e-7
):

    probabilities = torch.sigmoid(
        logits
    )

    predictions = (
        probabilities >= threshold
    ).float()

    predictions = predictions.flatten(1)
    targets = targets.flatten(1)

    intersection = (
        predictions * targets
    ).sum(dim=1)

    prediction_sum = (
        predictions.sum(dim=1)
    )

    target_sum = (
        targets.sum(dim=1)
    )

    union = (
        prediction_sum
        + target_sum
        - intersection
    )

    per_image_iou = (
        intersection + epsilon
    ) / (
        union + epsilon
    )

    per_image_dice = (
        2 * intersection + epsilon
    ) / (
        prediction_sum
        + target_sum
        + epsilon
    )

    return {
        "per_image_iou_sum":
            per_image_iou.sum().item(),

        "per_image_dice_sum":
            per_image_dice.sum().item(),

        "intersection":
            intersection.sum().item(),

        "union":
            union.sum().item(),

        "prediction_sum":
            prediction_sum.sum().item(),

        "target_sum":
            target_sum.sum().item(),

        "num_images":
            targets.shape[0]
    }

In [35]:
# ============================================================
# PHASE 2 — Encoder Adapter Experiment
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = DEVICE

print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: NVIDIA GeForce RTX 4090


In [39]:
# ============================================================
# Experiment definition
# ============================================================

# The main focus is EWI1S because decoder-only adaptation showed
# the largest failure there despite a very good AgriCLIP box.
#
# Fair comparison:
# A) Decoder-only: existing VegAnn -> EWI1S decoder checkpoint
# B) Decoder + lightweight encoder feature adapter:
#    same VegAnn starting decoder + same EWI1S training data
#
# Training uses GT boxes, as in the original fine-tuning pipeline.
# Final evaluation uses the AgriCLIP-generated K=32 box.

ADAPTER_TRAIN_DATASET = "EWI1S"
ADAPTER_NUM_EPOCHS = 10
ADAPTER_LEARNING_RATE = 1e-4
ADAPTER_WEIGHT_DECAY = 1e-4
ADAPTER_PATIENCE = 3

TOP_K = 32
BOX_MARGIN = 25
MASK_THRESHOLD = 0.5
ADAPTER_BOTTLENECK = 32

EWI1S_DECODER_ONLY_CHECKPOINT = (
    DOMAIN_FT_ROOT / "VegAnn_to_EWI1S" / "best_mask_decoder.pt"
)

ADAPTER_EXPERIMENT_DIR = (
    DOMAIN_FT_ROOT / "EWI1S_EncoderAdapter"
)
ADAPTER_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

ADAPTER_CHECKPOINT = (
    ADAPTER_EXPERIMENT_DIR / "best_encoder_adapter.pt"
)

print("Training dataset:", ADAPTER_TRAIN_DATASET)
print("Training images:", len(datasets_train["EWI1S"]))
print("Validation images:", len(datasets_val["EWI1S"]))
print("Decoder-only checkpoint:", EWI1S_DECODER_ONLY_CHECKPOINT)
print("Adapter checkpoint:", ADAPTER_CHECKPOINT)


Training dataset: EWI1S
Training images: 70
Validation images: 8
Decoder-only checkpoint: /home/zahranadeem/thesis/sam2_domain_finetuning/VegAnn_to_EWI1S/best_mask_decoder.pt
Adapter checkpoint: /home/zahranadeem/thesis/sam2_domain_finetuning/EWI1S_EncoderAdapter/best_encoder_adapter.pt


In [37]:
# ============================================================
# Load a decoder checkpoint into a fresh SAM2 model
# ============================================================

def create_sam2_from_decoder_checkpoint(checkpoint_path):
    model = Sam2Model.from_pretrained(MODEL_NAME)

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False
    )

    if isinstance(checkpoint, dict) and "mask_decoder_state_dict" in checkpoint:
        decoder_state = checkpoint["mask_decoder_state_dict"]
    elif isinstance(checkpoint, dict):
        decoder_state = checkpoint
    else:
        raise TypeError(
            f"Unsupported checkpoint structure: {type(checkpoint)}"
        )

    model.mask_decoder.load_state_dict(
        decoder_state,
        strict=True
    )

    model = model.to(DEVICE)
    return model, checkpoint

print("Decoder checkpoint loader ready.")


Decoder checkpoint loader ready.


In [40]:
# ============================================================
# Verify the existing decoder-only EWI1S checkpoint
# ============================================================

if not EWI1S_DECODER_ONLY_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Missing decoder-only EWI1S checkpoint:\n{EWI1S_DECODER_ONLY_CHECKPOINT}"
    )

baseline_model, baseline_ckpt = create_sam2_from_decoder_checkpoint(
    EWI1S_DECODER_ONLY_CHECKPOINT
)

print("Decoder-only checkpoint loaded.")
print(
    "Checkpoint epoch:",
    baseline_ckpt.get("epoch", "N/A") if isinstance(baseline_ckpt, dict) else "N/A"
)
print(
    "Checkpoint best IoU:",
    baseline_ckpt.get("best_mean_iou", "N/A") if isinstance(baseline_ckpt, dict) else "N/A"
)

del baseline_model
torch.cuda.empty_cache()


[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

Decoder-only checkpoint loaded.
Checkpoint epoch: 7
Checkpoint best IoU: 0.29224663347892094


In [41]:
# ============================================================
# Lightweight encoder feature adapter
# ============================================================

class SAM2FeatureAdapter(nn.Module):
    """
    Lightweight residual adapter applied to frozen SAM2
    image-encoder features before the mask decoder.

    The original encoder weights remain frozen.
    Only this adapter and the mask decoder are trained.
    """

    def __init__(self, channels, bottleneck=32):
        super().__init__()
        self.channels = channels
        self.bottleneck = bottleneck
        self.down = nn.Conv2d(channels, bottleneck, kernel_size=1, bias=True)
        self.act = nn.GELU()
        self.up = nn.Conv2d(bottleneck, channels, kernel_size=1, bias=True)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        residual = self.up(self.act(self.down(x)))
        return x + residual

print("SAM2 feature adapter class ready.")


SAM2 feature adapter class ready.


In [43]:
# ============================================================
# Probe SAM2 multi-scale image features
# ============================================================

adapter_probe_loader = DataLoader(
    datasets_train["EWI1S"],
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=sam2_collate_fn
)

probe_batch = next(iter(adapter_probe_loader))
probe_inputs = prepare_sam2_batch(probe_batch)

probe_model, _ = create_sam2_from_decoder_checkpoint(
    VEGANN_SAM2_CHECKPOINT
)
probe_model.eval()

with torch.no_grad():
    probe_embeddings = probe_model.get_image_embeddings(
        probe_inputs["pixel_values"]
    )

# ------------------------------------------------------------
# Inspect returned SAM2 feature structure
# ------------------------------------------------------------

print("Type returned by get_image_embeddings:",
      type(probe_embeddings))

if isinstance(probe_embeddings, (list, tuple)):

    print("Number of feature levels:", len(probe_embeddings))

    for i, feat in enumerate(probe_embeddings):
        print(
            f"Feature {i}: "
            f"type={type(feat)}, "
            f"shape={tuple(feat.shape)}, "
            f"dtype={feat.dtype}, "
            f"device={feat.device}"
        )

else:
    print(
        "Embedding shape:",
        tuple(probe_embeddings.shape)
    )

del probe_model, probe_batch, probe_inputs, probe_embeddings
torch.cuda.empty_cache()

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

Type returned by get_image_embeddings: <class 'list'>
Number of feature levels: 3
Feature 0: type=<class 'torch.Tensor'>, shape=(1, 32, 256, 256), dtype=torch.float32, device=cuda:0
Feature 1: type=<class 'torch.Tensor'>, shape=(1, 64, 128, 128), dtype=torch.float32, device=cuda:0
Feature 2: type=<class 'torch.Tensor'>, shape=(1, 256, 64, 64), dtype=torch.float32, device=cuda:0


In [45]:
# ============================================================
# Parameter counts — SAM2 baseline
# ============================================================

count_model, _ = create_sam2_from_decoder_checkpoint(
    VEGANN_SAM2_CHECKPOINT
)

# Total SAM2 parameters
sam2_total_parameters = sum(
    p.numel() for p in count_model.parameters()
)

# Mask decoder parameters
decoder_parameters = sum(
    p.numel() for p in count_model.mask_decoder.parameters()
)

# Encoder parameters
encoder_parameters = (
    sam2_total_parameters - decoder_parameters
)

print("=" * 60)
print("SAM2 PARAMETER COUNT")
print("=" * 60)

print(
    f"SAM2 total parameters:       "
    f"{sam2_total_parameters:,}"
)

print(
    f"Encoder parameters:          "
    f"{encoder_parameters:,}"
)

print(
    f"Mask decoder parameters:     "
    f"{decoder_parameters:,}"
)

print(
    f"Decoder-only trainable:      "
    f"{decoder_parameters:,}"
)

print("=" * 60)

del count_model
torch.cuda.empty_cache()

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

SAM2 PARAMETER COUNT
SAM2 total parameters:       31,440,721
Encoder parameters:          27,225,612
Mask decoder parameters:     4,215,109
Decoder-only trainable:      4,215,109


In [47]:
# ============================================================
# Probe SAM2 multi-scale image features
# ============================================================

adapter_probe_loader = DataLoader(
    datasets_train["EWI1S"],
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=sam2_collate_fn
)

probe_batch = next(iter(adapter_probe_loader))
probe_inputs = prepare_sam2_batch(probe_batch)

probe_model, _ = create_sam2_from_decoder_checkpoint(
    VEGANN_SAM2_CHECKPOINT
)

probe_model.eval()

with torch.no_grad():
    probe_embeddings = probe_model.get_image_embeddings(
        probe_inputs["pixel_values"]
    )

print("Type returned:",
      type(probe_embeddings))

if isinstance(probe_embeddings, (list, tuple)):

    print("Number of feature levels:",
          len(probe_embeddings))

    for i, feat in enumerate(probe_embeddings):
        print(
            f"Feature {i}: "
            f"shape={tuple(feat.shape)}, "
            f"dtype={feat.dtype}, "
            f"device={feat.device}"
        )

else:

    print(
        "Embedding shape:",
        tuple(probe_embeddings.shape)
    )

del probe_model
del probe_batch
del probe_inputs
del probe_embeddings

torch.cuda.empty_cache()

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

Type returned: <class 'list'>
Number of feature levels: 3
Feature 0: shape=(1, 32, 256, 256), dtype=torch.float32, device=cuda:0
Feature 1: shape=(1, 64, 128, 128), dtype=torch.float32, device=cuda:0
Feature 2: shape=(1, 256, 64, 64), dtype=torch.float32, device=cuda:0


In [48]:
# ============================================================
# Encoder-adapter training function
# ============================================================

def train_one_epoch_encoder_adapter(model, adapter, loader, optimizer, scaler):
    model.train()
    adapter.train()

    for p in model.parameters():
        p.requires_grad = False
    for p in model.mask_decoder.parameters():
        p.requires_grad = True

    running_loss = 0.0
    running_bce = 0.0
    running_dice = 0.0

    for batch_index, batch in enumerate(loader):
        inputs = prepare_sam2_batch(batch)
        optimizer.zero_grad(set_to_none=True)

        with torch.no_grad():
            image_embeddings = model.get_image_embeddings(
                inputs["pixel_values"]
            )

        with torch.amp.autocast(
            "cuda", enabled=torch.cuda.is_available()
        ):
            adapted_embeddings = adapter(image_embeddings)

            outputs = model(
                image_embeddings=adapted_embeddings,
                input_boxes=inputs["input_boxes"],
                multimask_output=False
            )

            logits = extract_single_mask_logits(outputs.pred_masks)
            targets = inputs["labels"].float()

            while targets.ndim > 3:
                targets = targets[:, 0]

            targets = F.interpolate(
                targets.unsqueeze(1),
                size=logits.shape[-2:],
                mode="nearest"
            ).squeeze(1)
            targets = (targets > 0).float()

            loss, bce, dice = combined_segmentation_loss(
                logits, targets
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            list(model.mask_decoder.parameters()) + list(adapter.parameters()),
            max_norm=1.0
        )

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        running_bce += bce.item()
        running_dice += dice.item()

        if (batch_index + 1) % 100 == 0:
            print(
                f"Batch {batch_index + 1}/{len(loader)} | "
                f"Loss: {running_loss / (batch_index + 1):.4f}"
            )

    n = len(loader)
    return {
        "loss": running_loss / n,
        "bce": running_bce / n,
        "dice_loss": running_dice / n
    }

print("Encoder-adapter training function ready.")


Encoder-adapter training function ready.


In [49]:
# ============================================================
# Encoder-adapter validation function
# ============================================================

@torch.no_grad()
def validate_encoder_adapter(model, adapter, loader, threshold=0.5):
    model.eval()
    adapter.eval()

    running_loss = 0.0
    metric_totals = {
        "per_image_iou_sum": 0.0,
        "per_image_dice_sum": 0.0,
        "intersection": 0.0,
        "union": 0.0,
        "prediction_sum": 0.0,
        "target_sum": 0.0,
        "num_images": 0
    }

    for batch in loader:
        inputs = prepare_sam2_batch(batch)
        image_embeddings = model.get_image_embeddings(
            inputs["pixel_values"]
        )
        adapted_embeddings = adapter(image_embeddings)

        outputs = model(
            image_embeddings=adapted_embeddings,
            input_boxes=inputs["input_boxes"],
            multimask_output=False
        )

        logits = extract_single_mask_logits(outputs.pred_masks)
        targets = inputs["labels"].float()

        while targets.ndim > 3:
            targets = targets[:, 0]

        targets = F.interpolate(
            targets.unsqueeze(1),
            size=logits.shape[-2:],
            mode="nearest"
        ).squeeze(1)
        targets = (targets > 0).float()

        loss, _, _ = combined_segmentation_loss(logits, targets)
        running_loss += loss.item()

        batch_metrics = compute_batch_metrics(
            logits, targets, threshold=threshold
        )
        for key in metric_totals:
            if key in batch_metrics:
                metric_totals[key] += batch_metrics[key]

    n_images = metric_totals["num_images"]
    if n_images == 0:
        raise RuntimeError("No validation images were evaluated.")

    mean_iou = metric_totals["per_image_iou_sum"] / n_images
    mean_dice = metric_totals["per_image_dice_sum"] / n_images
    global_iou = metric_totals["intersection"] / (metric_totals["union"] + 1e-7)
    global_dice = (
        2 * metric_totals["intersection"]
        / (
            metric_totals["prediction_sum"]
            + metric_totals["target_sum"]
            + 1e-7
        )
    )

    return {
        "loss": running_loss / len(loader),
        "mean_iou": mean_iou,
        "mean_dice": mean_dice,
        "global_iou": global_iou,
        "global_dice": global_dice
    }

print("Encoder-adapter validation function ready.")


Encoder-adapter validation function ready.


In [51]:
# ============================================================
# Inspect the actual SAM2 feature outputs
# ============================================================

probe_model, _ = create_sam2_from_decoder_checkpoint(
    VEGANN_SAM2_CHECKPOINT
)
probe_model.eval()

probe_batch = next(iter(adapter_probe_loader))
probe_inputs = prepare_sam2_batch(probe_batch)

with torch.no_grad():
    probe_embeddings = probe_model.get_image_embeddings(
        probe_inputs["pixel_values"]
    )

print("Type:", type(probe_embeddings))

if isinstance(probe_embeddings, (list, tuple)):

    print("Number of feature levels:", len(probe_embeddings))

    for i, feat in enumerate(probe_embeddings):
        print(
            f"Level {i}: "
            f"shape={tuple(feat.shape)}"
        )

else:
    print(
        "Embedding shape:",
        tuple(probe_embeddings.shape)
    )

del probe_model, probe_batch, probe_inputs, probe_embeddings
torch.cuda.empty_cache()

[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

Type: <class 'list'>
Number of feature levels: 3
Level 0: shape=(1, 32, 256, 256)
Level 1: shape=(1, 64, 128, 128)
Level 2: shape=(1, 256, 64, 64)


In [52]:
# ============================================================
# Multi-scale SAM2 feature adapter
# ============================================================

import torch
import torch.nn as nn


class SAM2MultiScaleFeatureAdapter(nn.Module):
    """
    Lightweight residual adapters for the three SAM2 Hiera
    image-feature levels.

    Input:
        [
            [B, 32, 256, 256],
            [B, 64, 128, 128],
            [B, 256, 64, 64]
        ]

    Output:
        Same structure and shapes.

    The residual formulation starts close to the identity:
        output = feature + adapter(feature)

    This allows the frozen SAM2 encoder representation to be
    adjusted for the EWI1S domain without fine-tuning the
    encoder itself.
    """

    def __init__(
        self,
        channels=(32, 64, 256),
        bottleneck=16
    ):
        super().__init__()

        self.channels = tuple(channels)
        self.bottleneck = bottleneck

        self.adapters = nn.ModuleList()

        for c in self.channels:
            self.adapters.append(
                nn.Sequential(
                    nn.Conv2d(
                        c,
                        bottleneck,
                        kernel_size=1,
                        bias=False
                    ),
                    nn.GELU(),
                    nn.Conv2d(
                        bottleneck,
                        c,
                        kernel_size=1,
                        bias=True
                    )
                )
            )

        # Start with almost exactly the original frozen features.
        # This makes the adapter a residual correction rather
        # than replacing the pretrained representation.
        for adapter_block in self.adapters:
            nn.init.zeros_(adapter_block[-1].weight)
            nn.init.zeros_(adapter_block[-1].bias)

    def forward(self, features):

        if not isinstance(features, (list, tuple)):
            raise TypeError(
                "Expected SAM2 features as a list/tuple, got "
                f"{type(features)}"
            )

        if len(features) != len(self.adapters):
            raise ValueError(
                f"Expected {len(self.adapters)} feature levels, "
                f"got {len(features)}"
            )

        adapted_features = []

        for feature, adapter_block in zip(
            features,
            self.adapters
        ):
            adapted = feature + adapter_block(feature)
            adapted_features.append(adapted)

        return adapted_features

In [53]:
# ============================================================
# SAM2 Hiera feature configuration
# ============================================================

FEATURE_CHANNELS = (32, 64, 256)

encoder_adapter = SAM2MultiScaleFeatureAdapter(
    channels=FEATURE_CHANNELS,
    bottleneck=ADAPTER_BOTTLENECK
).to(DEVICE)

adapter_parameters = sum(
    p.numel()
    for p in encoder_adapter.parameters()
)

print("SAM2 feature channels:", FEATURE_CHANNELS)
print("Adapter bottleneck:", ADAPTER_BOTTLENECK)
print("Adapter parameters:", f"{adapter_parameters:,}")

SAM2 feature channels: (32, 64, 256)
Adapter bottleneck: 32
Adapter parameters: 22,880


In [54]:
# ============================================================
# Train decoder + encoder adapter on the FULL EWI1S train pool
# ============================================================

train_loader_adapter = DataLoader(
    datasets_train["EWI1S"],
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=sam2_collate_fn
)

val_loader_adapter = DataLoader(
    datasets_val["EWI1S"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=sam2_collate_fn
)

adapter_model, starting_checkpoint = create_sam2_from_decoder_checkpoint(
    VEGANN_SAM2_CHECKPOINT
)

for p in adapter_model.parameters():
    p.requires_grad = False
for p in adapter_model.mask_decoder.parameters():
    p.requires_grad = True

adapter = SAM2FeatureAdapter(
    channels=FEATURE_CHANNELS,
    bottleneck=ADAPTER_BOTTLENECK
).to(DEVICE)

optimizer = torch.optim.AdamW(
    list(adapter_model.mask_decoder.parameters()) + list(adapter.parameters()),
    lr=ADAPTER_LEARNING_RATE,
    weight_decay=ADAPTER_WEIGHT_DECAY
)

scaler = torch.amp.GradScaler(
    "cuda", enabled=torch.cuda.is_available()
)

history = {
    "epoch": [],
    "train_loss": [],
    "train_bce": [],
    "train_dice_loss": [],
    "val_loss": [],
    "mean_iou": [],
    "mean_dice": [],
    "global_iou": [],
    "global_dice": []
}

best_iou = -1.0
epochs_without_improvement = 0

for epoch in range(ADAPTER_NUM_EPOCHS):
    print("\n" + "=" * 70)
    print(
        f"EWI1S ENCODER ADAPTER — "
        f"EPOCH {epoch + 1}/{ADAPTER_NUM_EPOCHS}"
    )
    print("=" * 70)

    train_results = train_one_epoch_encoder_adapter(
        adapter_model, adapter, train_loader_adapter, optimizer, scaler
    )

    val_results = validate_encoder_adapter(
        adapter_model, adapter, val_loader_adapter, MASK_THRESHOLD
    )

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(train_results["loss"])
    history["train_bce"].append(train_results["bce"])
    history["train_dice_loss"].append(train_results["dice_loss"])
    history["val_loss"].append(val_results["loss"])
    history["mean_iou"].append(val_results["mean_iou"])
    history["mean_dice"].append(val_results["mean_dice"])
    history["global_iou"].append(val_results["global_iou"])
    history["global_dice"].append(val_results["global_dice"])

    print(f"Train Loss: {train_results['loss']:.4f}")
    print(f"Val Loss:   {val_results['loss']:.4f}")
    print(f"Mean IoU:   {100 * val_results['mean_iou']:.2f}%")
    print(f"Mean Dice:  {100 * val_results['mean_dice']:.2f}%")

    if val_results["mean_iou"] > best_iou:
        best_iou = val_results["mean_iou"]
        epochs_without_improvement = 0

        torch.save(
            {
                "experiment_name": "EWI1S_EncoderAdapter",
                "epoch": epoch + 1,
                "model_name": MODEL_NAME,
                "starting_decoder_checkpoint": str(VEGANN_SAM2_CHECKPOINT),
                "train_dataset": "EWI1S",
                "train_samples": len(datasets_train["EWI1S"]),
                "val_samples": len(datasets_val["EWI1S"]),
                "learning_rate": ADAPTER_LEARNING_RATE,
                "weight_decay": ADAPTER_WEIGHT_DECAY,
                "adapter_channels": FEATURE_CHANNELS,
                "adapter_bottleneck": ADAPTER_BOTTLENECK,
                "mask_decoder_state_dict": adapter_model.mask_decoder.state_dict(),
                "encoder_adapter_state_dict": adapter.state_dict(),
                "best_mean_iou": best_iou,
                "history": history
            },
            ADAPTER_CHECKPOINT
        )

        print("\n*** BEST ENCODER-ADAPTER CHECKPOINT SAVED ***")
        print(ADAPTER_CHECKPOINT)
    else:
        epochs_without_improvement += 1
        print(
            f"No improvement for {epochs_without_improvement} epoch(s)."
        )

    pd.DataFrame(history).to_csv(
        ADAPTER_EXPERIMENT_DIR / "training_history.csv",
        index=False
    )

    if epochs_without_improvement >= ADAPTER_PATIENCE:
        print("\nEncoder-adapter early stopping triggered.")
        break

history_df_adapter = pd.DataFrame(history)

print("\n" + "=" * 70)
print("ENCODER-ADAPTER TRAINING COMPLETE")
print("=" * 70)
print(f"Best EWI1S validation IoU: {100 * best_iou:.2f}%")
print("Checkpoint:", ADAPTER_CHECKPOINT)


[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

TypeError: unsupported operand type(s) for %: 'tuple' and 'int'

In [55]:
# ============================================================
# GT-box comparison: decoder-only vs decoder + adapter
# ============================================================

baseline_model, _ = create_sam2_from_decoder_checkpoint(
    EWI1S_DECODER_ONLY_CHECKPOINT
)
baseline_val = validate(
    baseline_model, val_loader_adapter, MASK_THRESHOLD
)

adapter_eval_model, _ = create_sam2_from_decoder_checkpoint(
    VEGANN_SAM2_CHECKPOINT
)
adapter_checkpoint = torch.load(
    ADAPTER_CHECKPOINT, map_location="cpu", weights_only=False
)
adapter_eval_model.mask_decoder.load_state_dict(
    adapter_checkpoint["mask_decoder_state_dict"], strict=True
)
adapter_eval = SAM2FeatureAdapter(
    channels=adapter_checkpoint["adapter_channels"],
    bottleneck=adapter_checkpoint["adapter_bottleneck"]
).to(DEVICE)
adapter_eval.load_state_dict(
    adapter_checkpoint["encoder_adapter_state_dict"], strict=True
)
adapter_val = validate_encoder_adapter(
    adapter_eval_model, adapter_eval, val_loader_adapter, MASK_THRESHOLD
)

comparison = pd.DataFrame([
    {
        "method": "Decoder-only",
        "trainable_parameters": sum(
            p.numel() for p in baseline_model.mask_decoder.parameters()
        ),
        "IoU_%": 100 * baseline_val["mean_iou"],
        "Dice_%": 100 * baseline_val["mean_dice"]
    },
    {
        "method": "Decoder + Encoder Adapter",
        "trainable_parameters": (
            sum(p.numel() for p in adapter_eval_model.mask_decoder.parameters())
            + sum(p.numel() for p in adapter_eval.parameters())
        ),
        "IoU_%": 100 * adapter_val["mean_iou"],
        "Dice_%": 100 * adapter_val["mean_dice"]
    }
])

display(comparison.round(3))
comparison.to_csv(
    ADAPTER_EXPERIMENT_DIR / "gt_box_decoder_vs_adapter.csv",
    index=False
)

del baseline_model, adapter_eval_model, adapter_eval
torch.cuda.empty_cache()


[transformers] You are using a model of type `sam2_video` to instantiate a model of type `sam2`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/309 [00:00<?, ?it/s]

NameError: name 'validate' is not defined

In [ ]:
# ============================================================
# AgriCLIP + DINO setup for final pipeline evaluation
# ============================================================

import sys

AGRICLIP_PATH = (
    "/home/zahranadeem/thesis/agriclip_sam2/"
    "AgriCLIP/AgriCLIP_alignment"
)
if AGRICLIP_PATH not in sys.path:
    sys.path.insert(0, AGRICLIP_PATH)

from TextToConcept import TextToConcept
from LinearAligner import LinearAligner
from torchvision import transforms

ALIGNER_PATH = Path(
    "/home/zahranadeem/thesis/agriclip_sam2/AgriCLIP/"
    "Weights/Aligned_Models/Agri_Dino_aligner_DDEF_CDEF.pth"
)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

dino_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

dino_model = torch.hub.load(
    "facebookresearch/dino:main",
    "dino_resnet50",
    pretrained=True
)
dino_model.eval().to(device)
for p in dino_model.parameters():
    p.requires_grad = False

text_to_concept = TextToConcept(dino_model, "dino_resnet50")
text_to_concept.load_linear_aligner(str(ALIGNER_PATH))
aligner = text_to_concept.linear_aligner
aligner.eval()

captured_feature_map = None
def capture_layer4_output(module, inputs, output):
    global captured_feature_map
    captured_feature_map = output.detach()

hook_handle = dino_model.layer4.register_forward_hook(
    capture_layer4_output
)

@torch.no_grad()
def extract_dino_patch_features(pil_image):
    global captured_feature_map
    captured_feature_map = None
    image_tensor = dino_transform(pil_image).unsqueeze(0).to(device)
    _ = dino_model(image_tensor)
    if captured_feature_map is None:
        raise RuntimeError("The layer4 hook did not capture a feature map.")
    if captured_feature_map.ndim != 4:
        raise RuntimeError(
            f"Expected feature map [B,C,H,W], got {captured_feature_map.shape}"
        )
    _, channels, grid_h, grid_w = captured_feature_map.shape
    patch_features = (
        captured_feature_map[0]
        .permute(1, 2, 0)
        .reshape(grid_h * grid_w, channels)
        .contiguous()
    )
    return patch_features, grid_h, grid_w

@torch.no_grad()
def align_patch_features(patch_features):
    aligned_features = aligner.get_aligned_representation(
        patch_features.float().to(DEVICE)
    )
    return F.normalize(aligned_features, dim=1)

POSITIVE_PROMPTS = [
    "vegetation", "green vegetation", "crop plants",
    "crop vegetation", "agricultural plants", "green plants"
]
NEGATIVE_PROMPTS = [
    "soil", "bare soil", "ground", "bare ground",
    "non-vegetated ground", "agricultural soil"
]

@torch.no_grad()
def encode_prompt_set(prompts):
    embeddings = []
    for prompt in prompts:
        embedding = text_to_concept.encode_text([[prompt]])
        if isinstance(embedding, np.ndarray):
            embedding = torch.from_numpy(embedding)
        embedding = embedding.detach().float().to(device).reshape(-1)
        embeddings.append(embedding)
    embeddings = F.normalize(torch.stack(embeddings), dim=1)
    mean_embedding = F.normalize(embeddings.mean(dim=0), dim=0)
    return embeddings, mean_embedding

positive_text_embeddings, positive_embedding = encode_prompt_set(POSITIVE_PROMPTS)
negative_text_embeddings, negative_embedding = encode_prompt_set(NEGATIVE_PROMPTS)
vegetation_embedding = F.normalize(
    positive_embedding - negative_embedding, dim=0
)

@torch.no_grad()
def generate_agriclip_box(image, top_k=32, margin=25):
    patch_features, grid_h, grid_w = extract_dino_patch_features(image)
    aligned_features = align_patch_features(patch_features)
    patch_scores = aligned_features @ vegetation_embedding
    scores = patch_scores.cpu().numpy()
    topk_indices = np.argsort(scores)[::-1][:top_k]

    W, H = image.size
    patch_h = H / grid_h
    patch_w = W / grid_w
    boxes = []
    for idx in topk_indices:
        r = idx // grid_w
        c = idx % grid_w
        boxes.append([
            int(c * patch_w), int(r * patch_h),
            int((c + 1) * patch_w), int((r + 1) * patch_h)
        ])
    boxes = np.asarray(boxes)
    xmin = max(0, boxes[:, 0].min() - margin)
    ymin = max(0, boxes[:, 1].min() - margin)
    xmax = min(W - 1, boxes[:, 2].max() + margin)
    ymax = min(H - 1, boxes[:, 3].max() + margin)
    return np.array([xmin, ymin, xmax, ymax], dtype=np.int32), boxes, scores, topk_indices

sam2_processor = Sam2Processor.from_pretrained(MODEL_NAME)

print("AgriCLIP + DINO pipeline ready.")


In [ ]:
# ============================================================
# Standard decoder-only and adapter SAM2 multi-mask predictors
# ============================================================

@torch.no_grad()
def sam2_predict_box_multimask(model, image, box, threshold=0.5):
    inputs = sam2_processor(
        images=image,
        input_boxes=[[box.tolist()]],
        return_tensors="pt"
    )
    inputs = {
        k: v.to(DEVICE) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }
    outputs = model(**inputs, multimask_output=True)
    best_idx = outputs.iou_scores[0, 0].argmax().item()
    logits = outputs.pred_masks[0, 0, best_idx]
    probability = torch.sigmoid(logits)
    height = inputs["original_sizes"][0, 0].item()
    width = inputs["original_sizes"][0, 1].item()
    probability = F.interpolate(
        probability[None, None], size=(height, width),
        mode="bilinear", align_corners=False
    )[0, 0]
    prediction = (probability > threshold).cpu().numpy().astype(np.uint8)
    confidence = outputs.iou_scores[0, 0, best_idx].float().item()
    return prediction, confidence, best_idx

@torch.no_grad()
def adapter_sam2_predict_box_multimask(model, adapter, image, box, threshold=0.5):
    inputs = sam2_processor(
        images=image,
        input_boxes=[[box.tolist()]],
        return_tensors="pt"
    )
    inputs = {
        k: v.to(DEVICE) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }
    image_embeddings = model.get_image_embeddings(inputs["pixel_values"])
    adapted_embeddings = adapter(image_embeddings)
    outputs = model(
        image_embeddings=adapted_embeddings,
        input_boxes=inputs["input_boxes"],
        multimask_output=True
    )
    best_idx = outputs.iou_scores[0, 0].argmax().item()
    logits = outputs.pred_masks[0, 0, best_idx]
    probability = torch.sigmoid(logits)
    height = inputs["original_sizes"][0, 0].item()
    width = inputs["original_sizes"][0, 1].item()
    probability = F.interpolate(
        probability[None, None], size=(height, width),
        mode="bilinear", align_corners=False
    )[0, 0]
    prediction = (probability > threshold).cpu().numpy().astype(np.uint8)
    confidence = outputs.iou_scores[0, 0, best_idx].float().item()
    return prediction, confidence, best_idx

print("SAM2 predictors ready.")


In [ ]:
# ============================================================
# Evaluation helpers and datasets
# ============================================================

def load_evaluation_item(record, dataset_name):
    if dataset_name == "VegAnn":
        sample = record["hf_dataset"][record["index"]]
        image = sample["image"]
        gt_mask = binary_mask_from_array(sample["mask"])
    elif dataset_name == "ReLeaf":
        image, gt_mask = load_releaf_image_mask(record["image_path"])
    elif dataset_name == "WeedsGalore":
        image, gt_mask = load_weedsgalore_image_mask(record)
    elif dataset_name == "EWI1S":
        image, gt_mask = load_ewis1_image_mask(record)
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")
    if not isinstance(image, Image.Image):
        image = Image.fromarray(np.asarray(image))
    return image.convert("RGB"), np.asarray(gt_mask).astype(bool)

evaluation_datasets = {
    "VegAnn": vegann_val_records,
    "ReLeaf": releaf_val_pairs,
    "WeedsGalore": weedsgalore_val_pairs,
    "EWI1S": ewis1_val_pairs
}

print({k: len(v) for k, v in evaluation_datasets.items()})

def _metric_dict(prediction, gt_mask):
    prediction = prediction.astype(bool)
    if prediction.shape != gt_mask.shape:
        prediction = cv2.resize(
            prediction.astype(np.uint8),
            (gt_mask.shape[1], gt_mask.shape[0]),
            interpolation=cv2.INTER_NEAREST
        ).astype(bool)
    intersection = np.logical_and(prediction, gt_mask).sum()
    union = np.logical_or(prediction, gt_mask).sum()
    pred_area = prediction.sum()
    gt_area = gt_mask.sum()
    iou = intersection / union if union > 0 else 0.0
    dice = (2 * intersection / (pred_area + gt_area)
            if pred_area + gt_area > 0 else 0.0)
    fp = np.logical_and(prediction, ~gt_mask).sum()
    fn = np.logical_and(~prediction, gt_mask).sum()
    precision = intersection / (intersection + fp) if intersection + fp > 0 else 0.0
    recall = intersection / (intersection + fn) if intersection + fn > 0 else 0.0
    return iou, dice, precision, recall


In [ ]:
# ============================================================
# Final AgriCLIP -> SAM2 evaluation
# ============================================================

@torch.no_grad()
def evaluate_decoder_only(checkpoint_path, dataset_records, dataset_name):
    model, _ = create_sam2_from_decoder_checkpoint(checkpoint_path)
    model.eval()
    values = []
    for idx, record in enumerate(dataset_records):
        try:
            image, gt_mask = load_evaluation_item(record, dataset_name)
            if gt_mask.sum() == 0:
                continue
            box, _, _, _ = generate_agriclip_box(image, TOP_K, BOX_MARGIN)
            pred, conf, best_idx = sam2_predict_box_multimask(
                model, np.array(image), box, MASK_THRESHOLD
            )
            values.append((*_metric_dict(pred, gt_mask), conf, best_idx))
            if (idx + 1) % 25 == 0 or idx == 0:
                print(f"Decoder-only | {dataset_name} | {idx + 1}/{len(dataset_records)}")
        except Exception as e:
            print(f"ERROR decoder-only | {dataset_name} | image {idx}: {type(e).__name__}: {e}")
    result = {
        "checkpoint": "EWI1S_DecoderOnly",
        "dataset": dataset_name,
        "images_evaluated": len(values),
        "IoU_%": 100 * np.mean([x[0] for x in values]) if values else np.nan,
        "Dice_%": 100 * np.mean([x[1] for x in values]) if values else np.nan,
        "Precision_%": 100 * np.mean([x[2] for x in values]) if values else np.nan,
        "Recall_%": 100 * np.mean([x[3] for x in values]) if values else np.nan,
        "mean_confidence": np.mean([x[4] for x in values]) if values else np.nan
    }
    del model
    torch.cuda.empty_cache()
    return result

@torch.no_grad()
def evaluate_adapter(adapter_checkpoint_path, dataset_records, dataset_name):
    checkpoint = torch.load(adapter_checkpoint_path, map_location="cpu", weights_only=False)
    model, _ = create_sam2_from_decoder_checkpoint(VEGANN_SAM2_CHECKPOINT)
    model.mask_decoder.load_state_dict(checkpoint["mask_decoder_state_dict"], strict=True)
    adapter = SAM2FeatureAdapter(
        checkpoint["adapter_channels"], checkpoint["adapter_bottleneck"]
    ).to(DEVICE)
    adapter.load_state_dict(checkpoint["encoder_adapter_state_dict"], strict=True)
    model.eval(); adapter.eval()
    values = []
    for idx, record in enumerate(dataset_records):
        try:
            image, gt_mask = load_evaluation_item(record, dataset_name)
            if gt_mask.sum() == 0:
                continue
            box, _, _, _ = generate_agriclip_box(image, TOP_K, BOX_MARGIN)
            pred, conf, best_idx = adapter_sam2_predict_box_multimask(
                model, adapter, np.array(image), box, MASK_THRESHOLD
            )
            values.append((*_metric_dict(pred, gt_mask), conf, best_idx))
            if (idx + 1) % 25 == 0 or idx == 0:
                print(f"Adapter | {dataset_name} | {idx + 1}/{len(dataset_records)}")
        except Exception as e:
            print(f"ERROR adapter | {dataset_name} | image {idx}: {type(e).__name__}: {e}")
    result = {
        "checkpoint": "EWI1S_DecoderPlusAdapter",
        "dataset": dataset_name,
        "images_evaluated": len(values),
        "IoU_%": 100 * np.mean([x[0] for x in values]) if values else np.nan,
        "Dice_%": 100 * np.mean([x[1] for x in values]) if values else np.nan,
        "Precision_%": 100 * np.mean([x[2] for x in values]) if values else np.nan,
        "Recall_%": 100 * np.mean([x[3] for x in values]) if values else np.nan,
        "mean_confidence": np.mean([x[4] for x in values]) if values else np.nan
    }
    del model, adapter
    torch.cuda.empty_cache()
    return result

print("Final evaluation functions ready.")


In [ ]:
# ============================================================
# PRIMARY RESULT — Decoder-only vs Encoder Adapter on EWI1S
# ============================================================

if not ADAPTER_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Missing adapter checkpoint. Run training first: {ADAPTER_CHECKPOINT}"
    )

phase2_primary_results = [
    evaluate_decoder_only(
        EWI1S_DECODER_ONLY_CHECKPOINT,
        evaluation_datasets["EWI1S"],
        "EWI1S"
    ),
    evaluate_adapter(
        ADAPTER_CHECKPOINT,
        evaluation_datasets["EWI1S"],
        "EWI1S"
    )
]

phase2_primary_df = pd.DataFrame(phase2_primary_results)
display(phase2_primary_df.round(3))

phase2_primary_df.to_csv(
    ADAPTER_EXPERIMENT_DIR / "PRIMARY_EWI1S_AGRICLIP_COMPARISON.csv",
    index=False
)


In [ ]:
# ============================================================
# Optional robustness check — both methods across all four datasets
# ============================================================

phase2_cross_domain = []
for dataset_name, records in evaluation_datasets.items():
    phase2_cross_domain.append(
        evaluate_decoder_only(
            EWI1S_DECODER_ONLY_CHECKPOINT, records, dataset_name
        )
    )
    phase2_cross_domain.append(
        evaluate_adapter(
            ADAPTER_CHECKPOINT, records, dataset_name
        )
    )

phase2_cross_domain_df = pd.DataFrame(phase2_cross_domain)
display(phase2_cross_domain_df.round(3))

phase2_cross_domain_df.to_csv(
    ADAPTER_EXPERIMENT_DIR / "CROSS_DOMAIN_AGRICLIP_COMPARISON.csv",
    index=False
)
